In [40]:
from pathlib import Path
import zipfile
import pandas as pd
import hashlib
import io

In [7]:
BASE_DIR = Path("/data/kaliani/datasets/all_doc_judge") 
TARGET = "regions.csv"

In [21]:

def read_regions_schema_from_zip(zip_path: Path) -> dict:
    """Return schema info for regions.csv inside a zip archive."""
    with zipfile.ZipFile(zip_path, "r") as zf:
        # знайдемо regions.csv навіть якщо він у підпапці
        candidates = [n for n in zf.namelist() if n.endswith(f"/{TARGET}") or n == TARGET]
        if not candidates:
            return {"zip": zip_path.name, "found": False}

        member = candidates[0]  # беремо перший збіг
        with zf.open(member) as f:

            df0 = pd.read_csv(f, nrows=0)

        cols = list(df0.columns)
        return {
            "zip": zip_path.name,
            "found": True,
            "member_path": member,
            "n_cols": len(cols),
            "columns": cols,
        }

# 1) зібрати схеми з усіх архівів
zip_files = sorted(BASE_DIR.glob("*.zip"))
schemas = [read_regions_schema_from_zip(zp) for zp in zip_files]
schemas_df = pd.DataFrame(schemas)

# display(schemas_df[["zip","found","member_path","n_cols"]])

found_df = schemas_df[schemas_df["found"]].copy()
if found_df.empty:
    raise RuntimeError("Не знайшов regions.csv в жодному zip.")

baseline_cols = found_df.iloc[0]["columns"]
baseline_zip = found_df.iloc[0]["zip"]

def diff_cols(cols, baseline):
    cols_set = set(cols)
    base_set = set(baseline)
    return {
        "same_order": cols == baseline,
        "same_set": cols_set == base_set,
        "added": sorted(list(cols_set - base_set)),
        "removed": sorted(list(base_set - cols_set)),
    }

diff_rows = []
for _, row in found_df.iterrows():
    d = diff_cols(row["columns"], baseline_cols)
    diff_rows.append({
        "zip": row["zip"],
        "same_set": d["same_set"],
        "same_order": d["same_order"],
        "added": d["added"],
        "removed": d["removed"],
    })

diff_df = pd.DataFrame(diff_rows).sort_values(["same_set","same_order","zip"])
display(diff_df)

print(f"Baseline: {baseline_zip}")

,zip,same_set,same_order,added,removed
0,edrsr_data_2006.zip,True,True,[],[]
1,edrsr_data_2007.zip,True,True,[],[]
2,edrsr_data_2008.zip,True,True,[],[]
3,edrsr_data_2009.zip,True,True,[],[]
4,edrsr_data_2010.zip,True,True,[],[]
5,edrsr_data_2011.zip,True,True,[],[]
6,edrsr_data_2012.zip,True,True,[],[]
7,edrsr_data_2013.zip,True,True,[],[]
8,edrsr_data_2014.zip,True,True,[],[]
9,edrsr_data_2015.zip,True,True,[],[]


Baseline: edrsr_data_2006.zip


In [38]:
TARGET = "regions.csv"

In [39]:
def _fingerprint(columns: list[str], dtypes: dict) -> str:
    """
    Stable fingerprint for 'schema' = columns in order + dtypes.
    """
    payload = {
        "columns": columns,
        "dtypes": {c: str(dtypes.get(c, "")) for c in columns},  # keep same order
    }
    s = str(payload).encode("utf-8")
    return hashlib.sha256(s).hexdigest()[:16]

def read_regions_schema_from_zip(zip_path: Path) -> dict:
    """Return schema info for TARGET inside a zip archive + row count."""
    with zipfile.ZipFile(zip_path, "r") as zf:
        candidates = [n for n in zf.namelist() if n.endswith(f"/{TARGET}") or n == TARGET]
        if not candidates:
            return {"zip": zip_path.name, "found": False}

        member = candidates[0]

        # 1) read header/schema
        with zf.open(member) as f:
            df0 = pd.read_csv(f, sep="\t", nrows=0)

        cols = list(df0.columns)
        dtypes = df0.dtypes.astype(str).to_dict()

        # 2) count rows (fast, streaming)
        with zf.open(member) as f:
            # count all physical lines (including header)
            # decode lazily via TextIOWrapper
            text_f = io.TextIOWrapper(f, encoding="utf-8", errors="replace", newline="")
            total_lines = sum(1 for _ in text_f)

        # rows in data = lines - 1 header (but not below 0)
        n_rows = max(total_lines - 1, 0)

        return {
            "zip": zip_path.name,
            "found": True,
            "member_path": member,
            "n_cols": len(cols),
            "n_rows": n_rows,
            "columns": cols,
            "dtypes": dtypes,
            "header_checksum": _fingerprint(cols, dtypes),
        }


# 1) collect schemas
zip_files = sorted(BASE_DIR.glob("*.zip"))
schemas = [read_regions_schema_from_zip(zp) for zp in zip_files]
schemas_df = pd.DataFrame(schemas)

found_df = schemas_df[schemas_df["found"]].copy()
if found_df.empty:
    raise RuntimeError("Не знайшов regions.csv в жодному zip.")

# baseline
baseline_cols = found_df.iloc[0]["columns"]
baseline_dtypes = found_df.iloc[0]["dtypes"]
baseline_zip = found_df.iloc[0]["zip"]
baseline_checksum = found_df.iloc[0]["header_checksum"]


def diff_schema(cols, dtypes, base_cols, base_dtypes):
    cols_set = set(cols)
    base_set = set(base_cols)

    same_order = cols == base_cols
    same_set = cols_set == base_set

    # dtype diffs only for common columns
    common = [c for c in base_cols if c in dtypes]
    dtype_diff = {
        c: {"baseline": str(base_dtypes.get(c)), "current": str(dtypes.get(c))}
        for c in common
        if str(base_dtypes.get(c)) != str(dtypes.get(c))
    }

    return {
        "same_order": same_order,
        "same_set": same_set,
        "identical": same_order and same_set and (len(dtype_diff) == 0),
        "added": sorted(cols_set - base_set),
        "removed": sorted(base_set - cols_set),
        "dtype_mismatch": len(dtype_diff) > 0,
        "dtype_diff": dtype_diff,  # dict; can be big but useful
    }


diff_rows = []
for _, row in found_df.iterrows():
    d = diff_schema(
        row["columns"], row["dtypes"],
        baseline_cols, baseline_dtypes
    )

    diff_rows.append({
        "zip": row["zip"],
        "member_path": row["member_path"],
        "n_cols": row["n_cols"],
        "n_rows": row["n_rows"], 
        "same_set": d["same_set"],
        "same_order": d["same_order"],
        "dtype_mismatch": d["dtype_mismatch"],
        "identical": d["identical"],
        "added": d["added"],
        "removed": d["removed"],
        "dtype_diff": d["dtype_diff"],
        "header_checksum": row["header_checksum"],
        "checksum_equals_baseline": row["header_checksum"] == baseline_checksum,
    })

diff_df = pd.DataFrame(diff_rows).sort_values(
    ["identical", "same_set", "same_order", "dtype_mismatch", "zip"],
    ascending=[True, True, True, True, True]
)

display(diff_df)
print(f"Baseline: {baseline_zip} (checksum={baseline_checksum})")

,zip,member_path,n_cols,n_rows,same_set,same_order,dtype_mismatch,identical,added,removed,dtype_diff,header_checksum,checksum_equals_baseline
0,edrsr_data_2006.zip,regions.csv,2,31,True,True,False,True,[],[],{},dac12f79e18bed8c,True
1,edrsr_data_2007.zip,regions.csv,2,31,True,True,False,True,[],[],{},dac12f79e18bed8c,True
2,edrsr_data_2008.zip,regions.csv,2,31,True,True,False,True,[],[],{},dac12f79e18bed8c,True
3,edrsr_data_2009.zip,regions.csv,2,31,True,True,False,True,[],[],{},dac12f79e18bed8c,True
4,edrsr_data_2010.zip,regions.csv,2,31,True,True,False,True,[],[],{},dac12f79e18bed8c,True
5,edrsr_data_2011.zip,regions.csv,2,31,True,True,False,True,[],[],{},dac12f79e18bed8c,True
6,edrsr_data_2012.zip,regions.csv,2,31,True,True,False,True,[],[],{},dac12f79e18bed8c,True
7,edrsr_data_2013.zip,regions.csv,2,31,True,True,False,True,[],[],{},dac12f79e18bed8c,True
8,edrsr_data_2014.zip,regions.csv,2,31,True,True,False,True,[],[],{},dac12f79e18bed8c,True
9,edrsr_data_2015.zip,regions.csv,2,31,True,True,False,True,[],[],{},dac12f79e18bed8c,True


Baseline: edrsr_data_2006.zip (checksum=dac12f79e18bed8c)


In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

BASE_DIR = Path("/data/kaliani/datasets/all_doc_judge")

user = "kaliani"        
password = "123qwe123" 
host = "localhost"
port = "5432"
db = "mydb"

engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}")

In [5]:
from pathlib import Path
import zipfile
import pandas as pd

BASE_DIR = Path("/data/kaliani/datasets/all_doc_judge")

def iter_zip_files(base_dir: Path):
    for p in sorted(base_dir.rglob("*.zip")):
        yield p

def read_documents_csv_from_zip(zip_path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        # пробуємо знайти documents.csv будь-де всередині
        candidates = [n for n in z.namelist() if n.lower().endswith("documents.csv")]
        if not candidates:
            raise FileNotFoundError("documents.csv not found in zip")
        csv_name = candidates[0]
        with z.open(csv_name) as f:
            # якщо у тебе буває інша кодировка — можна спробувати 'utf-8-sig' або 'cp1251'
            return pd.read_csv(f)


In [6]:
import numpy as np

DOCS_SCHEMA = {
    # col: (type, max_len)
    "doc_id": ("int", None),
    "court_code": ("int", None),
    "judgment_code": ("int", None),
    "justice_kind": ("int", None),
    "category_code": ("int", None),

    "cause_num": ("str", 128),
    "adjudication_date": ("str", 50),
    "receipt_date": ("str", 50),
    "judge": ("str", 50),
    "doc_url": ("str", 128),

    "status": ("int", None),
    "date_publ": ("str", 50),
}

def normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    # на випадок різних назв у csv
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    return df

def validate_df(df: pd.DataFrame, schema=DOCS_SCHEMA):
    """
    return: (ok_df, errors_df)
    errors_df має по рядку на помилку
    """
    df = normalize_cols(df)

    # missing / extra columns
    missing = [c for c in schema.keys() if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # беремо тільки потрібні колонки (щоб зайві не заважали)
    df2 = df[list(schema.keys())].copy()

    errors = []

    # row_num для трейсингу
    df2["_row_num"] = np.arange(1, len(df2)+1)

    for col, (tp, max_len) in schema.items():
        s = df2[col]

        if tp == "int":
            # допускаємо пусті/NaN
            bad_mask = s.notna() & pd.to_numeric(s, errors="coerce").isna()
            if bad_mask.any():
                for rn, val in df2.loc[bad_mask, ["_row_num", col]].itertuples(index=False):
                    errors.append({"row_num": int(rn), "error_type": "bad_int", "column": col, "value": str(val)})
            # приводимо
            df2[col] = pd.to_numeric(s, errors="coerce").astype("Int64")

        elif tp == "str":
            # приводимо до str, але NaN залишаємо NaN
            s2 = s.where(s.notna(), None)
            s2 = s2.map(lambda x: str(x) if x is not None else None)

            if max_len is not None:
                lens = s2.map(lambda x: len(x) if x is not None else 0)
                bad_mask = lens > max_len
                if bad_mask.any():
                    for rn, val, ln in df2.loc[bad_mask, ["_row_num", col]].assign(_len=lens[bad_mask]).itertuples(index=False):
                        errors.append({
                            "row_num": int(rn),
                            "error_type": "too_long",
                            "column": col,
                            "value": (val[:200] + "…") if isinstance(val, str) and len(val) > 200 else str(val),
                            "length": int(ln),
                            "max_len": int(max_len),
                        })

            df2[col] = s2

    errors_df = pd.DataFrame(errors)
    ok_df = df2.drop(columns=["_row_num"])
    return ok_df, errors_df


In [7]:
from sqlalchemy import create_engine

# engine вже має бути створений як раніше
# engine = create_engine(...)

def write_ok_rows(ok_df: pd.DataFrame, zip_path: Path, year: int):
    out = ok_df.copy()
    out.insert(0, "source_file", str(zip_path))
    out.insert(1, "source_year", int(year))
    out.insert(2, "row_num", range(1, len(out)+1))
    out.to_sql("documents_import", engine, schema="public", if_exists="append", index=False, method="multi", chunksize=5000)

def write_errors(errors_df: pd.DataFrame, raw_df: pd.DataFrame, zip_path: Path, year: int):
    if errors_df.empty:
        return
    # додаємо raw_row json для конкретних row_num
    raw = normalize_cols(raw_df)
    raw["_row_num"] = np.arange(1, len(raw)+1)
    raw_map = raw.set_index("_row_num").to_dict(orient="index")

    err_out = []
    for r in errors_df.to_dict(orient="records"):
        rn = int(r["row_num"])
        err_out.append({
            "source_file": str(zip_path),
            "source_year": int(year),
            "row_num": rn,
            "error_type": r["error_type"],
            "error_message": f'{r.get("column","")}: {r.get("value","")}',
            "raw_row": raw_map.get(rn, {}),
        })

    pd.DataFrame(err_out).to_sql(
        "documents_import_errors",
        engine,
        schema="public",
        if_exists="append",
        index=False,
        method="multi",
        chunksize=2000
    )


In [8]:
import re

def infer_year_from_path(p: Path) -> int | None:
    m = re.search(r"(19|20)\d{2}", str(p))
    return int(m.group(0)) if m else None

report = []

for zip_path in iter_zip_files(BASE_DIR):
    year = infer_year_from_path(zip_path)

    try:
        raw_df = read_documents_csv_from_zip(zip_path)
        ok_df, errors_df = validate_df(raw_df)

        # пишемо OK рядки
        write_ok_rows(ok_df, zip_path, year or -1)

        # пишемо помилки
        write_errors(errors_df, raw_df, zip_path, year or -1)

        report.append({
            "zip": str(zip_path),
            "year": year,
            "rows_total": len(raw_df),
            "rows_ok": len(ok_df) - (0),  # ok_df вже “нормалізований”
            "errors": int(len(errors_df)),
            "status": "done"
        })

    except Exception as e:
        report.append({
            "zip": str(zip_path),
            "year": year,
            "rows_total": None,
            "rows_ok": None,
            "errors": None,
            "status": f"failed: {type(e).__name__}: {e}"
        })

pd.DataFrame(report)


,zip,year,rows_total,rows_ok,errors,status
0,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2006,None,None,None,failed: ParserError: Error tokenizing data. C ...
1,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2007,None,None,None,failed: ParserError: Error tokenizing data. C ...
2,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2008,None,None,None,failed: ParserError: Error tokenizing data. C ...
3,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2009,None,None,None,failed: ParserError: Error tokenizing data. C ...
4,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2010,None,None,None,failed: ParserError: Error tokenizing data. C ...
5,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2011,None,None,None,failed: ParserError: Error tokenizing data. C ...
6,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2012,None,None,None,failed: ParserError: Error tokenizing data. C ...
7,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2013,None,None,None,failed: ParserError: Error tokenizing data. C ...
8,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2014,None,None,None,failed: ParserError: Error tokenizing data. C ...
9,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2015,None,None,None,failed: ParserError: Error tokenizing data. C ...


In [ ]:
from pathlib import Path
import zipfile, csv, io
import pandas as pd

def sniff_delimiter(sample_text: str) -> str:
    # найчастіші варіанти для таких датасетів
    candidates = [",", ";", "\t", "|"]
    # простий хак: беремо той, що найчастіше зустрічається в першому рядку
    first_line = sample_text.splitlines()[0] if sample_text else ""
    counts = {d: first_line.count(d) for d in candidates}
    # якщо всі 0 — хай буде comma
    return max(counts, key=counts.get) if max(counts.values()) > 0 else ","

def read_documents_csv_from_zip(zip_path: Path, encoding="utf-8"):
    with zipfile.ZipFile(zip_path, "r") as z:
        candidates = [n for n in z.namelist() if n.lower().endswith("documents.csv")]
        if not candidates:
            raise FileNotFoundError("documents.csv not found in zip")
        csv_name = candidates[0]

        with z.open(csv_name) as f:
            raw = f.read()

    # пробуємо декілька енкодінгів (бо UA дані часто cp1251)
    text = None
    for enc in [encoding, "utf-8-sig", "cp1251", "latin1"]:
        try:
            text = raw.decode(enc)
            used_enc = enc
            break
        except UnicodeDecodeError:
            continue
    if text is None:
        raise UnicodeDecodeError("Cannot decode file with tried encodings")

    # sniff delimiter по шапці
    delim = sniff_delimiter(text[:5000])

    bad_lines = []
    def bad_line_handler(bad_line):
        # bad_line = list[str] (поля) або рядок — залежить від pandas версії
        bad_lines.append(bad_line)
        return None  # пропускаємо рядок

    df = pd.read_csv(
        io.StringIO(text),
        sep=delim,
        engine="python",          # ключове
        quoting=csv.QUOTE_MINIMAL,
        on_bad_lines=bad_line_handler,  # pandas>=1.3
        dtype=str,                # читаємо все як str, потім валідуємо
    )

    return df, {"encoding": used_enc, "delimiter": delim, "bad_lines_cnt": len(bad_lines), "bad_lines_sample": bad_lines[:3]}


report = []

for zip_path in iter_zip_files(BASE_DIR):
    year = infer_year_from_path(zip_path)

    try:
        raw_df, meta = read_documents_csv_from_zip(zip_path)

        ok_df, errors_df = validate_df(raw_df)   # твоя перевірка довжин/інтів

        write_ok_rows(ok_df, zip_path, year or -1)
        write_errors(errors_df, raw_df, zip_path, year or -1)

        report.append({
            "zip": str(zip_path),
            "year": year,
            "rows_total": len(raw_df),
            "rows_ok": len(ok_df),
            "errors": int(len(errors_df)),
            "bad_lines_cnt": meta["bad_lines_cnt"],
            "encoding": meta["encoding"],
            "delimiter": meta["delimiter"],
            "status": "done"
        })

    except Exception as e:
        report.append({
            "zip": str(zip_path),
            "year": year,
            "status": f"failed: {type(e).__name__}: {e}"
        })

pd.DataFrame(report)



In [1]:

# =========================
# 1) Imports & DB config
# =========================
from pathlib import Path
import re, io, zipfile
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

BASE_DIR = Path("/data/kaliani/datasets/all_doc_judge")

DB_USER = "kaliani"
DB_PASS = "123qwe123"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "mydb"

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")


# =========================
# 3) Helpers: zip iteration + year infer
# =========================
def iter_zip_files(base_dir: Path):
    for p in sorted(base_dir.rglob("*.zip")):
        yield p

def infer_year_from_path(p: Path) -> int | None:
    m = re.search(r"(19|20)\d{2}", str(p))
    return int(m.group(0)) if m else None

# =========================
# 4) Schema validation (based on your table screenshot)
# =========================
DOCS_SCHEMA = {
    "doc_id": ("int", None),
    "court_code": ("int", None),
    "judgment_code": ("int", None),
    "justice_kind": ("int", None),
    "category_code": ("int", None),

    "cause_num": ("str", 128),
    "adjudication_date": ("str", 50),
    "receipt_date": ("str", 50),
    "judge": ("str", 50),
    "doc_url": ("str", 128),

    "status": ("int", None),
    "date_publ": ("str", 50),
}

def normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    return df

def validate_df(df: pd.DataFrame, schema=DOCS_SCHEMA):
    df = normalize_cols(df)

    missing = [c for c in schema.keys() if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df2 = df[list(schema.keys())].copy()
    df2["_row_num"] = np.arange(1, len(df2)+1)

    errors = []

    for col, (tp, max_len) in schema.items():
        s = df2[col]

        if tp == "int":
            # detect non-numeric values
            bad_mask = s.notna() & (pd.to_numeric(s, errors="coerce").isna())
            if bad_mask.any():
                bad_rows = df2.loc[bad_mask, ["_row_num", col]]
                for rn, val in bad_rows.itertuples(index=False):
                    errors.append({"row_num": int(rn), "error_type": "bad_int", "column": col, "value": str(val)})

            df2[col] = pd.to_numeric(s, errors="coerce").astype("Int64")

        elif tp == "str":
            s2 = s.where(s.notna(), None)
            s2 = s2.map(lambda x: str(x) if x is not None else None)

            if max_len is not None:
                lens = s2.map(lambda x: len(x) if x is not None else 0)
                bad_mask = lens > max_len
                if bad_mask.any():
                    bad_rows = df2.loc[bad_mask, ["_row_num", col]].copy()
                    bad_rows["_len"] = lens[bad_mask].values
                    for rn, val, ln in bad_rows.itertuples(index=False):
                        errors.append({
                            "row_num": int(rn),
                            "error_type": "too_long",
                            "column": col,
                            "value": (val[:200] + "…") if isinstance(val, str) and len(val) > 200 else str(val),
                            "length": int(ln),
                            "max_len": int(max_len),
                        })

            df2[col] = s2

    errors_df = pd.DataFrame(errors)
    ok_df = df2.drop(columns=["_row_num"])
    return ok_df, errors_df

# =========================
# 5) Streaming read from zip (no RAM blow) + delimiter try
# =========================
DELIMS_TO_TRY = ["\t", ";", ",", "|"]

def find_documents_csv_name(z: zipfile.ZipFile) -> str:
    candidates = [n for n in z.namelist() if n.lower().endswith("documents.csv")]
    if not candidates:
        raise FileNotFoundError("documents.csv not found in zip")
    return candidates[0]

def chunk_reader_from_zip(zip_path: Path, sep: str, chunksize: int):
    with zipfile.ZipFile(zip_path, "r") as z:
        csv_name = find_documents_csv_name(z)
        raw_f = z.open(csv_name, "r")
        # decode as stream; replace bad chars
        txt = io.TextIOWrapper(raw_f, encoding="utf-8", errors="replace", newline="")
        reader = pd.read_csv(
            txt,
            sep=sep,
            engine="python",
            dtype=str,
            chunksize=chunksize,
            on_bad_lines="skip",
        )
        for chunk in reader:
            yield chunk, csv_name

def pick_working_delimiter(zip_path: Path, chunksize=2000):
    """
    Try delimiters and pick the one that yields the most columns in the first chunk
    and has at least 5 columns (heuristic).
    """
    best = None
    best_cols = -1
    best_csv_name = None

    for sep in DELIMS_TO_TRY:
        try:
            gen = chunk_reader_from_zip(zip_path, sep=sep, chunksize=chunksize)
            first_chunk, csv_name = next(gen)
            ncols = first_chunk.shape[1]
            if ncols > best_cols:
                best = sep
                best_cols = ncols
                best_csv_name = csv_name
        except StopIteration:
            continue
        except Exception:
            continue

    if best is None:
        raise ValueError("Could not read documents.csv with any delimiter")

    # sanity threshold (your table has ~12 cols)
    if best_cols < 8:
        raise ValueError(f"Delimiter detection failed (best ncols={best_cols}). Best sep={repr(best)}")

    return best, best_cols, best_csv_name

# =========================
# 6) DB writers
# =========================
def write_ok_rows(ok_df: pd.DataFrame, zip_path: Path, year: int, start_row_num: int):
    out = ok_df.copy()
    out.insert(0, "source_file", str(zip_path))
    out.insert(1, "source_year", int(year))
    out.insert(2, "row_num", range(start_row_num, start_row_num + len(out)))
    out.to_sql("documents_import", engine, schema="public", if_exists="append",
               index=False, method="multi", chunksize=5000)

def write_errors(errors_df: pd.DataFrame, raw_df: pd.DataFrame, zip_path: Path, year: int, start_row_num: int):
    if errors_df.empty:
        return

    raw = normalize_cols(raw_df)
    raw["_row_num"] = np.arange(start_row_num, start_row_num + len(raw))
    raw_map = raw.set_index("_row_num").to_dict(orient="index")

    err_out = []
    for r in errors_df.to_dict(orient="records"):
        rn = int(r["row_num"]) + (start_row_num - 1)  # adjust within-file row numbering
        err_out.append({
            "source_file": str(zip_path),
            "source_year": int(year),
            "row_num": rn,
            "error_type": r["error_type"],
            "error_message": f'{r.get("column","")}: {r.get("value","")}',
            "raw_row": raw_map.get(rn, {}),
        })

    pd.DataFrame(err_out).to_sql("documents_import_errors", engine, schema="public",
                                 if_exists="append", index=False, method="multi", chunksize=2000)

# =========================
# 7) Process one zip (streaming)
# =========================
def process_zip(zip_path: Path, year: int | None, chunksize=20000):
    year = year or -1

    sep, ncols, csv_name = pick_working_delimiter(zip_path, chunksize=min(5000, chunksize))

    total = 0
    total_ok = 0
    total_errs = 0
    row_cursor = 1  # row_num within this source file

    gen = chunk_reader_from_zip(zip_path, sep=sep, chunksize=chunksize)

    for chunk, _ in gen:
        total += len(chunk)

        ok_df, errors_df = validate_df(chunk)

        # write OK
        write_ok_rows(ok_df, zip_path, year, start_row_num=row_cursor)

        # write ERRORS (with raw row snapshot)
        write_errors(errors_df, chunk, zip_path, year, start_row_num=row_cursor)

        total_ok += len(ok_df)
        total_errs += len(errors_df)

        row_cursor += len(chunk)

    return {
        "zip": str(zip_path),
        "year": year,
        "csv_name": csv_name,
        "delimiter": repr(sep),
        "ncols_first_chunk": ncols,
        "rows_total": total,
        "rows_ok": total_ok,
        "errors": total_errs,
        "status": "done",
    }

# =========================
# 8) Run all zips
# =========================
report = []

for zip_path in iter_zip_files(BASE_DIR):
    y = infer_year_from_path(zip_path)
    try:
        rep = process_zip(zip_path, y, chunksize=20000)
        report.append(rep)
        print("OK:", zip_path.name, "rows:", rep["rows_total"], "errs:", rep["errors"], "sep:", rep["delimiter"])
    except Exception as e:
        report.append({
            "zip": str(zip_path),
            "year": y,
            "status": f"failed: {type(e).__name__}: {e}"
        })
        print("FAIL:", zip_path.name, type(e).__name__, e)

report_df = pd.DataFrame(report)
report_df


OK: edrsr_data_2006.zip rows: 340132 errs: 0 sep: '\t'
OK: edrsr_data_2007.zip rows: 1084475 errs: 0 sep: '\t'
OK: edrsr_data_2008.zip rows: 2185272 errs: 0 sep: '\t'
OK: edrsr_data_2009.zip rows: 3539588 errs: 0 sep: '\t'
OK: edrsr_data_2010.zip rows: 5869688 errs: 0 sep: '\t'
OK: edrsr_data_2011.zip rows: 7128333 errs: 0 sep: '\t'
OK: edrsr_data_2012.zip rows: 6903092 errs: 0 sep: '\t'
OK: edrsr_data_2013.zip rows: 7704258 errs: 0 sep: '\t'
OK: edrsr_data_2014.zip rows: 5714427 errs: 0 sep: '\t'
OK: edrsr_data_2015.zip rows: 12275301 errs: 0 sep: '\t'
OK: edrsr_data_2016.zip rows: 8773262 errs: 0 sep: '\t'
OK: edrsr_data_2017.zip rows: 7377311 errs: 0 sep: '\t'
OK: edrsr_data_2018.zip rows: 7474808 errs: 0 sep: '\t'
OK: edrsr_data_2019.zip rows: 7778328 errs: 0 sep: '\t'
OK: edrsr_data_2020.zip rows: 7192344 errs: 0 sep: '\t'
OK: edrsr_data_2021.zip rows: 8397897 errs: 0 sep: '\t'
OK: edrsr_data_2022.zip rows: 5823306 errs: 0 sep: '\t'
OK: edrsr_data_2023.zip rows: 7836259 errs: 0 se

,zip,year,csv_name,delimiter,ncols_first_chunk,rows_total,rows_ok,errors,status
0,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2006,documents.csv,'\t',12,340132,340132,0,done
1,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2007,documents.csv,'\t',12,1084475,1084475,0,done
2,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2008,documents.csv,'\t',12,2185272,2185272,0,done
3,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2009,documents.csv,'\t',12,3539588,3539588,0,done
4,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2010,documents.csv,'\t',12,5869688,5869688,0,done
5,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2011,documents.csv,'\t',12,7128333,7128333,0,done
6,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2012,documents.csv,'\t',12,6903092,6903092,0,done
7,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2013,documents.csv,'\t',12,7704258,7704258,0,done
8,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2014,documents.csv,'\t',12,5714427,5714427,0,done
9,/data/kaliani/datasets/all_doc_judge/edrsr_dat...,2015,documents.csv,'\t',12,12275301,12275301,0,done


In [41]:
BASE_DIR = Path("/data/kaliani/datasets/judges__list") 
TARGET = "main-spisok-suddiv-na-09-02-2026.xlsx"

In [108]:
df = pd.read_excel("/data/kaliani/datasets/judges__list/main-spisok-suddiv-na-09-02-2026.xlsx")

In [89]:
cols1 = {"№ з/п": "id", 
        "Номер досьє": "dos_id",
        "Прізвище, ім'я, по батькові осіб, призначених (обраних) на посаду судді\n(сортування за алфавітом)": "name", 
        "Стать": "sex",
        "Найменування суду, до якого призначено (обрано, переведено) особу": "court_name"}
df = df.rename(columns=cols1)

In [96]:
cols2 = {"Прізвище, ім'я, по батькові \n(сортування за алфавітом)": "name",
         "Найменування суду, до якого призначено (обрано) особу": "court_name_old"}

In [ ]:
df_other = pd.read_excel("/data/kaliani/datasets/judges__list/spisok-suddiv-na-12-06-2023.xlsx", header=1).rename(columns=cols2)

In [112]:
df_info = pd.read_excel("/data/kaliani/datasets/judges__list/spisok-suddiv-na-12-06-2023.xlsx", header=None, nrows=1)

In [115]:
df_info.iloc[0, 0]

'СПИСОК СУДДІВ СТАНОМ НА 12.06.2023'

In [107]:
df_other

,СПИСОК СУДДІВ СТАНОМ НА 12.06.2023,Unnamed: 1,Unnamed: 2
0,№ з/п,"Прізвище, ім'я, по батькові \n(сортування за а...","Найменування суду, до якого призначено (обрано..."
1,1,Аббасова Наталія Володимирівна,Шевченківський районний суд міста Києва
2,2,Абдукадирова Каріне Ескендерівна,Донецький окружний адміністративний суд
3,3,Аблов Євгеній Валерійович,Окружний адміністративний суд міста Києва
4,4,Аблова Юлія Юріївна,Комінтернівський районний суд Одеської області
...,...,...,...
5075,5075,Яцун Оксана Олександрівна,Великобілозерський районний суд Запорізької об...
5076,5076,Яцун Олександр Сергійович,Заводський районний суд міста Запоріжжя
5077,5077,Ячало Юрій Іванович,Великобагачанський районний суд Полтавської об...
5078,5078,Ященко Світлана Олександрівна,Комінтернівський районний суд міста Харкова


In [86]:
### якщо ПІБ відсутній повністю в першому дф
### якщо є ПІБ але інший суд
### якщо є ПІБ та Суд


df_other

,№ з/п,name,court_name
0,1,Аббасова Наталія Володимирівна,Шевченківський районний суд міста Києва
1,2,Абдукадирова Каріне Ескендерівна,Донецький окружний адміністративний суд
2,3,Аблов Євгеній Валерійович,Окружний адміністративний суд міста Києва
3,4,Аблова Юлія Юріївна,Комінтернівський районний суд Одеської області
4,5,Абрамов Петро Станіславович,Полтавський апеляційний суд
...,...,...,...
5074,5075,Яцун Оксана Олександрівна,Великобілозерський районний суд Запорізької об...
5075,5076,Яцун Олександр Сергійович,Заводський районний суд міста Запоріжжя
5076,5077,Ячало Юрій Іванович,Великобагачанський районний суд Полтавської об...
5077,5078,Ященко Світлана Олександрівна,Комінтернівський районний суд міста Харкова


In [99]:
for_ch = df.merge(df_other, left_on=["name", "court_name"], right_on=["name", "court_name_old"], how="right")

In [103]:
for_ch[(for_ch["dos_id"].notna()) & (for_ch["court_name"] == for_ch["court_name_old"])]

,id,dos_id,name,sex,court_name,№ з/п,court_name_old
0,1.0,6935.0,Аббасова Наталія Володимирівна,Ж,Шевченківський районний суд міста Києва,1,Шевченківський районний суд міста Києва
1,2.0,1237.0,Абдукадирова Каріне Ескендерівна,Ж,Донецький окружний адміністративний суд,2,Донецький окружний адміністративний суд
2,3.0,3431.0,Аблов Євгеній Валерійович,Ч,Окружний адміністративний суд міста Києва,3,Окружний адміністративний суд міста Києва
5,5.0,2122.0,Абухін Руслан Дмитрович,Ч,Приморський районний суд міста Одеси,6,Приморський районний суд міста Одеси
6,6.0,1080.0,Аверкова Вікторія Вікторівна,Ж,Окружний адміністративний суд міста Києва,7,Окружний адміністративний суд міста Києва
...,...,...,...,...,...,...,...
5073,4770.0,4336.0,Яцина Олександр Іванович,Ч,Новоушицький районний суд Хмельницької області,5074,Новоушицький районний суд Хмельницької області
5074,4771.0,4785.0,Яцун Оксана Олександрівна,Ж,Великобілозерський районний суд Запорізької об...,5075,Великобілозерський районний суд Запорізької об...
5075,4772.0,4534.0,Яцун Олександр Сергійович,Ч,Заводський районний суд міста Запоріжжя,5076,Заводський районний суд міста Запоріжжя
5076,4773.0,4400.0,Ячало Юрій Іванович,Ч,Великобагачанський районний суд Полтавської об...,5077,Великобагачанський районний суд Полтавської об...
